[Reference](https://medium.com/@pankaj_pandey/434c97cf1645)

# Hybrid retrieval is no longer optional
```
results = client.query.get(
    "Document", ["content"]
).with_hybrid(
    query="contract termination procedures",
    alpha=0.7  # 0 = pure BM25, 1 = pure vector
).with_limit(10).do()
```

# Self-correcting RAG with LangGraph

In [1]:
from langgraph.graph import StateGraph, START, END

# Each node is a function: retrieve(), grade_documents(),
# generate(), transform_query(), web_search()
# Full definitions in the CRAG tutorial linked below
workflow = StateGraph(GraphState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("rewrite_query", transform_query)
workflow.add_node("web_search", web_search)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents", decide_to_generate,
    {"rewrite_query": "rewrite_query", "generate": "generate"}
)
workflow.add_edge("rewrite_query", "web_search")
workflow.add_edge("web_search", "generate")
workflow.add_edge("generate", END)
app = workflow.compile()

# DSPy: programming agents, not prompting them

In [2]:
import dspy

class RAG(dspy.Module):
    def __init__(self):
        self.retrieve = dspy.Retrieve(k=5)
        self.respond = dspy.ChainOfThought(
            "context, question -> answer"
        )
    def forward(self, question):
        context = self.retrieve(question).passages
        return self.respond(
            context=context, question=question
        )
# Auto-optimize prompts on training data
optimizer = dspy.MIPROv2(metric=SemanticF1, auto="light")
optimized_rag = optimizer(RAG(), trainset=train_examples)

# LightRAG: graph retrieval that actually runs

In [3]:
from lightrag import LightRAG, QueryParam

rag = LightRAG(working_dir="./rag_index")
await rag.ainsert(documents)
# Four retrieval modes: naive, local, global, hybrid
result = await rag.aquery(
    "What are the main investment themes?",
    param=QueryParam(mode="hybrid")
)

# MCP: the universal agent-tool layer

In [4]:
from agents import Agent, Runner, FileSearchTool, WebSearchTool

agent = Agent(
    name="ResearchAgent",
    instructions="Search the knowledge base when you need "
                 "factual grounding. Use web search for "
                 "current events. Skip retrieval for "
                 "questions you can answer directly.",
    tools=[
        FileSearchTool(
            max_num_results=3,
            vector_store_ids=["vs_company_docs"]
        ),
        WebSearchTool(),
    ]
)
result = await Runner.run(agent, "What changed in our Q4 policy?")
print(result.final_output)

# How these compose together

- Hybrid retrieval as the search backbone.
- LangGraph or DSPy for orchestration and self-correction loops.
- LightRAG for domains where entity relationships matter.
- MCP as the standardized interface connecting everything.


# Semantic caching cuts the bill

In [5]:
from redisvl.extensions.llmcache import SemanticCache

llmcache = SemanticCache(
    name="rag_cache",
    redis_url="redis://localhost:6379",
    distance_threshold=0.1  # tighter = fewer false hits
)
# Check before calling the LLM
if cached := llmcache.check(prompt=user_query):
    return cached[0]["response"]
# On cache miss, generate and store
response = generate_answer(user_query)
llmcache.store(prompt=user_query, response=response)

# Measure it or it does not count

In [6]:
from ragas import evaluate
from datasets import Dataset

eval_data = Dataset.from_dict({
    "question": ["What is our refund policy?"],
    "answer": ["Refunds are processed within 14 days."],
    "contexts": [["Our refund policy allows returns within "
                   "14 business days of purchase."]],
    "ground_truth": ["Refunds take 14 business days."]
})
result = evaluate(eval_data)
print(result)
# {'context_precision': 0.92, 'faithfulness': 0.95,
#  'answer_relevancy': 0.88}